In [8]:
import pandas as pd
import joblib
import scipy.stats as stats
from sklearn.model_selection import train_test_split

In [9]:
def calculate_spearman(dataset, model_name):

    # Configurações
    MODEL_PATH = f"../candidate_models/{dataset}/{model_name}.joblib"
    DATA_PATH = f"../data/meta/meta_processed/meta_proc_{dataset}.csv"

    # Carregamento 
    model = joblib.load(MODEL_PATH)
    df = pd.read_csv(DATA_PATH)

    try:
        # Separa em atributos preditivos e atributos alvo
        target_cols = ['F1 (macro averaged by label)', 'Model Size Log'] # alvo
        X = df.drop(columns=target_cols + ['Model Size'], errors='ignore') # preditivos
        y = df[target_cols] # alvo
    except KeyError as e:
        print(f"Erro nas colunas: {e}")
        exit()

    # Usa apenas o conjunto de teste, pois os modelos 
    # foram treinados no conjunto de treino
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Inferência
    preds = model.predict(X_test) # faz a inferência

    # Caso retorne 2D (R2 e model size log) utiliza apenas R2
    y_pred_f1 = preds[:, 0] if preds.ndim > 1 else preds 
    y_real_f1 = y_test.iloc[:, 0].values # Pega os valores reais dos dados de teste

    # Cria um df com os resultados para comparação
    df_results = pd.DataFrame({'Real': y_real_f1, 'Pred': y_pred_f1}).reset_index()

    """ Realiza um teste de Spearman Global:
        Tenta ranquear os pipelines por ordem de desempenho, então 
        compara o ranking gerado com o real. O intuito é saber se um modelo
        consegue prever "o quão bom" é um pipeline, ou seja, é um recomendador
    """
    rho_global, _ = stats.spearmanr(df_results['Real'], df_results['Pred'])

    # Calcula Top-10% Overlap (Quantos dos top 10% reais estão no top 10% predito)
    k = int(len(df_results) * 0.10)
    top_real_idx = set(df_results.sort_values(by='Real', ascending=False).head(k).index)
    top_pred_idx = set(df_results.sort_values(by='Pred', ascending=False).head(k).index)
    overlap = len(top_real_idx.intersection(top_pred_idx)) / k

    # Calcula Regret (Perda de Performance)
    best_real_val = df_results['Real'].max()
    predicted_winner_real_val = df_results.loc[df_results['Pred'].idxmax(), 'Real']
    regret = best_real_val - predicted_winner_real_val

    print(f"Resultados para o dataset: {dataset}")
    print(f"Modelo usado: {model_name}")
    print(f"Spearman Global: {rho_global:.4f}")
    print(f"Overlap no Top {k} (Recall): {overlap:.2%}")
    print(f"Regret (Perda de F1): {regret:.4f}")
    print("\n---------------------------------\n")


#### **Como interpretar as métricas de avaliação:**
- Spearman Global: É a ordem que o modelo ranqueia os pipelines; quanto mais perto de 1, melhor o modelo sabe a "posição certa" que o pipleine deveria estar
- Overlap Top-k: Mesma coisa do Spearman global, porém dentro dos 10% melhores pipelines; prova se o modelo sabe escolher um pipeline dentro dos melhores
- Regret: É a "perda" ao confiar no modelo; é calculado a partir da diferença entre o melhor pipeline previsto e o melhor pipeline real

In [10]:
calculate_spearman(dataset="birds", model_name="mlp_native")
calculate_spearman(dataset="birds", model_name="xgboost_multi_output")
calculate_spearman(dataset="birds", model_name="lgbm_multi_output")

Resultados para o dataset: birds
Modelo usado: mlp_native
Spearman Global: 0.9569
Overlap no Top 2094 (Recall): 88.63%
Regret (Perda de F1): 0.1720

---------------------------------

Resultados para o dataset: birds
Modelo usado: xgboost_multi_output
Spearman Global: 0.9760
Overlap no Top 2094 (Recall): 92.41%
Regret (Perda de F1): 0.0720

---------------------------------

Resultados para o dataset: birds
Modelo usado: lgbm_multi_output
Spearman Global: 0.9703
Overlap no Top 2094 (Recall): 91.64%
Regret (Perda de F1): 0.0930

---------------------------------



In [11]:
calculate_spearman(dataset="medical", model_name="mlp_native")
calculate_spearman(dataset="medical", model_name="xgboost_multi_output")
calculate_spearman(dataset="medical", model_name="lgbm_multi_output")

Resultados para o dataset: medical
Modelo usado: mlp_native
Spearman Global: 0.9662
Overlap no Top 1472 (Recall): 68.00%
Regret (Perda de F1): 0.0730

---------------------------------

Resultados para o dataset: medical
Modelo usado: xgboost_multi_output
Spearman Global: 0.9857
Overlap no Top 1472 (Recall): 82.07%
Regret (Perda de F1): 0.0180

---------------------------------

Resultados para o dataset: medical
Modelo usado: lgbm_multi_output
Spearman Global: 0.9850
Overlap no Top 1472 (Recall): 81.66%
Regret (Perda de F1): 0.1530

---------------------------------



In [12]:
calculate_spearman(dataset="enron", model_name="mlp_native")
calculate_spearman(dataset="enron", model_name="xgboost_multi_output")
calculate_spearman(dataset="enron", model_name="lgbm_multi_output")

Resultados para o dataset: enron
Modelo usado: mlp_native
Spearman Global: 0.9244
Overlap no Top 1595 (Recall): 73.92%
Regret (Perda de F1): 0.0300

---------------------------------

Resultados para o dataset: enron
Modelo usado: xgboost_multi_output
Spearman Global: 0.9839
Overlap no Top 1595 (Recall): 86.96%
Regret (Perda de F1): 0.0070

---------------------------------

Resultados para o dataset: enron
Modelo usado: lgbm_multi_output
Spearman Global: 0.9823
Overlap no Top 1595 (Recall): 86.33%
Regret (Perda de F1): 0.0040

---------------------------------



In [13]:
calculate_spearman(dataset="scene", model_name="mlp_native")
calculate_spearman(dataset="scene", model_name="xgboost_multi_output")
calculate_spearman(dataset="scene", model_name="lgbm_multi_output")


Resultados para o dataset: scene
Modelo usado: mlp_native
Spearman Global: 0.9386
Overlap no Top 1549 (Recall): 77.40%
Regret (Perda de F1): 0.0640

---------------------------------

Resultados para o dataset: scene
Modelo usado: xgboost_multi_output
Spearman Global: 0.9801
Overlap no Top 1549 (Recall): 89.48%
Regret (Perda de F1): 0.0000

---------------------------------

Resultados para o dataset: scene
Modelo usado: lgbm_multi_output
Spearman Global: 0.9752
Overlap no Top 1549 (Recall): 87.22%
Regret (Perda de F1): 0.0180

---------------------------------



In [14]:
calculate_spearman(dataset="yeast", model_name="mlp_native")
calculate_spearman(dataset="yeast", model_name="xgboost_multi_output")
calculate_spearman(dataset="yeast", model_name="lgbm_multi_output")

Resultados para o dataset: yeast
Modelo usado: mlp_native
Spearman Global: 0.9129
Overlap no Top 1890 (Recall): 63.92%
Regret (Perda de F1): 0.0340

---------------------------------

Resultados para o dataset: yeast
Modelo usado: xgboost_multi_output
Spearman Global: 0.9792
Overlap no Top 1890 (Recall): 84.60%
Regret (Perda de F1): 0.0140

---------------------------------

Resultados para o dataset: yeast
Modelo usado: lgbm_multi_output
Spearman Global: 0.9755
Overlap no Top 1890 (Recall): 82.65%
Regret (Perda de F1): 0.0070

---------------------------------



### **Conclusão:**
#### Com estes resultados, é possível concluir que os modelos realmente conseguem prever o desempenho F1 de um pipeline. Para tirar a prova de conceito, um script de geração aleatória irá gerar milhares de pipelines (muitos ainda nunca vistos), e o modelo fará a inferência do F1 score e do tamanho do modelo. Após chegar em um F1 satisfatório, o pipeline selecionado será implementado e testado à mão, para termos a comparação da previsão com a realidade. 